# Cell 1: Imports

In [34]:
import sagemaker
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import (
    InputData, 
    S3DataSource, 
    SourceCode, 
    Compute,
    OutputDataConfig,
    StoppingCondition,
    MetricDefinition,
    StoppingCondition
)

In [35]:
# --- USER SETTINGS ---
my_job_name = "3DCNN-Exp-2nd_1000" 
my_bucket = "alexander-thesis-cslr"

In [36]:
# 1. Metric Definitions (The Magic Part) 🔍
# Adjust the 'Regex' to match EXACTLY what your train_sagemaker.py prints.
# Example assumption: Your script prints "Epoch: 1, Loss: 0.45, Acc: 0.82"
# Updated Metric Definitions (using r'' to fix SyntaxWarning)
metrics = [
    MetricDefinition(name='train:loss',  regex=r'\[Metrics\] Train Loss: ([0-9\.]+)'),
    MetricDefinition(name='train:top1',  regex=r'\[Metrics\] Train Top1: ([0-9\.]+)'),
    MetricDefinition(name='train:top5',  regex=r'\[Metrics\] Train Top5: ([0-9\.]+)'),
    MetricDefinition(name='train:top10', regex=r'\[Metrics\] Train Top10: ([0-9\.]+)'),
    MetricDefinition(name='val:loss',    regex=r'\[Metrics\] Val Loss: ([0-9\.]+)'),
    MetricDefinition(name='val:top1',    regex=r'\[Metrics\] Val Top1: ([0-9\.]+)'),
    MetricDefinition(name='val:top5',    regex=r'\[Metrics\] Val Top5: ([0-9\.]+)'),
    MetricDefinition(name='val:top10',   regex=r'\[Metrics\] Val Top10: ([0-9\.]+)')
]

In [37]:
# 1. Define your code location
# source_dir='.' works because your notebook is inside Thesis-CSLR
code_config = SourceCode(
    source_dir='./ASL',
    entry_script='train_sagemaker.py' 
)

In [38]:
# 3. Hardware Configuration
# 'instance_type' and count move to this new object
compute_config = Compute(
    instance_type="ml.g5.4xlarge",
    instance_count=1
)

# Cell 2: Setup

In [39]:
# 4. Data Config
data_source = S3DataSource(
    s3_uri=f"s3://{my_bucket}/data_tensors_1000",
    s3_data_type="S3Prefix",
    s3_data_distribution_type="FullyReplicated"
)

In [40]:
data_input = InputData(
    channel_name="training",
    data_source=data_source
)

In [41]:
# 5. Output Config
output_config = OutputDataConfig(
    s3_output_path=f"s3://{my_bucket}/experiments/output"
)

In [42]:
stop_condition = StoppingCondition(
    max_runtime_in_seconds=432000
)

# Cell 3: Define the Experiment

In [43]:
# 3. Initialize the Unified Trainer
# You now provide the image URI directly
trainer = ModelTrainer(
    training_image="763104351884.dkr.ecr.eu-north-1.amazonaws.com/pytorch-training:2.0-gpu-py310",
    role="arn:aws:iam::600889066998:role/SageMaker_role_Alexander_for_thesis_CLSR",
    base_job_name="Thesis-ASL-Exp-3DCNN-2nd_1000",
    source_code=code_config,      # script location
    compute=compute_config,
    output_data_config=output_config,
    hyperparameters={
        "epochs": "200",
        "batch-size": "11",
        "num-classes": "1000",
        "learning-rate": "0.00005",
        "experiment-name": my_job_name,
        "model-type": "r3d_18"
    },
    # Set FastFile globally for the algorithm
    training_input_mode="File",
    environment={"PYTHONUNBUFFERED": "1"},
    stopping_condition=stop_condition,
    tags=[
        {'key': 'Project', 'value': 'CSLR-Thesis'},
        {'key': 'Model',   'value': '3DCNN'},
        {'key': 'User',    'value': 'Alexander'}
    ],
    
).with_metric_definitions(metrics)


[02/03/26 12:08:10] INFO     Found credentials in environment variables.                        ]8;id=380909;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=658218;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/botocore/credentials.py#1252\1252]8;;\

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=498355;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=740358;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=599230;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=267957;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py#162\162]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=25944;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=984601;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#548\548]8;;\
                             763104351884.dkr.ecr.eu-north-1.amazonaws.com/pytorch-training:2.                     
                             0-gpu-py310                                                                           

# Cell 4: Launch!

In [ ]:
print(f"🚀 Launching Job with Monitoring: {my_job_name}")
training_job = trainer.train(
    input_data_config=[InputData(channel_name="training", data_source=data_source)],
    wait=True
)

print(f"✅ Job submitted! Check the 'Monitor' tab in the console.")

🚀 Launching Job with Monitoring: 3DCNN-Exp-2nd_1000


                    INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=569651;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=416864;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#92\92]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

[02/03/26 12:08:13] INFO     Creating training_job resource.                                     ]8;id=319921;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=871168;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35539\35539]8;;\

Output()

[02/03/26 12:17:31] INFO     Thesis-ASL-Exp-3DCNN-2nd-1000-20260203120810/algo-1-1770120541:     ]8;id=305383;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=903535;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35869\35869]8;;\
                             Starting training script                                                              

...

                    INFO     Thesis-ASL-Exp-3DCNN-2nd-1000-20260203120810/algo-1-1770120541:     ]8;id=471924;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=664809;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35869\35869]8;;\
                             Downloading:                                                                          
                             "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to                          
                             /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth                                

[02/03/26 12:17:37] INFO     Thesis-ASL-Exp-3DCNN-2nd-1000-20260203120810/algo-1-1770120541:     ]8;id=687141;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=996558;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35869\35869]8;;\
                             0%|          | 0.00/127M [00:00<?, ?B/s]#015  6%|▌         |                          
                             7.35M/127M [00:00<00:01, 76.7MB/s]#015 20%|██        | 25.9M/127M                     
                             [00:00<00:00, 146MB/s] #015 32%|███▏      | 41.4M/127M                                
                             [00:00<00:00, 153MB/s]#015 52%|█████▏    | 66.2M/127M [00:00<00:00,                   
                             192MB/s]#015 66%|██████▌   | 84.4M/127M [00:00<00:00, 100MB/s]#015                    
                             87%|████████▋ | 111M/127M [00:00<00:00, 137MB/s]                                      
                             #015100%|██████████| 127M/127M [00:00<00:00, 144MB/s]                                 

                    INFO     Thesis-ASL-Exp-3DCNN-2nd-1000-20260203120810/algo-1-1770120541:     ]8;id=681612;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=352671;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35869\35869]8;;\
                             --- Epoch 1/200 ---                                                                   

                    INFO     Thesis-ASL-Exp-3DCNN-2nd-1000-20260203120810/algo-1-1770120541:     ]8;id=105649;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=994932;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35869\35869]8;;\
                             🔥 Warmup Epoch 1/5: LR set to 1.04e-05                                               

[02/03/26 12:22:21] INFO     Thesis-ASL-Exp-3DCNN-2nd-1000-20260203120810/algo-1-1770120541:     ]8;id=893457;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=81913;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35869\35869]8;;\
                             Train Ep 1:   0%|          | 0/817 [00:00<?, ?it/s]Train Ep:                          
                             [1][204/817] Time: 285s (7.9 img/s) | Loss: 7.0597 | Top1: 0.18% |                    
                             Top5: 0.58% | Top10: 1.02%                                                            